# Feature Engineering

* Back joints (hip_center and shoulder_center)
* 8 joint angles
* 13 axis angles

### Imports

In [7]:
import numpy as np
import pandas as pd
from pathlib import Path

### Configuration

In [8]:
IN_DIR = Path("../data/processed/keypoints_combined")
OUT_DIR = Path("../data/processed/keypoints_combined_angles")

EPS = 1e-8

JOINTS = [
    "left_shoulder", "right_shoulder",
    "left_elbow", "right_elbow",
    "left_wrist", "right_wrist",
    "left_hip", "right_hip",
    "left_knee", "right_knee",
    "left_ankle", "right_ankle",
]

### Angle calculation

In [9]:
def angle_abc(points_a, points_b, points_c):
    u = points_a - points_b
    v = points_c - points_b
    du = np.linalg.norm(u, axis=1)
    dv = np.linalg.norm(v, axis=1)
    denom = du * dv
    dot = np.einsum("ij,ij->i", u, v)

    out = np.full(points_a.shape[0], np.nan, dtype=points_a.dtype)
    ok = (denom > EPS) & np.isfinite(denom) & np.isfinite(dot)

    cos_theta = np.empty_like(dot)
    cos_theta[ok] = dot[ok] / denom[ok]
    cos_theta[ok] = np.clip(cos_theta[ok], -1.0, 1.0)
    out[ok] = np.arccos(cos_theta[ok])
    return out


def angle_segment_to_axis(p0, p1, axis_unit):
    v = p1 - p0
    dv = np.linalg.norm(v, axis=1)
    dot = v @ axis_unit

    out = np.full(p0.shape[0], np.nan, dtype=p0.dtype)
    ok = (dv > EPS) & np.isfinite(dv) & np.isfinite(dot)

    cos_theta = np.empty(p0.shape[0], dtype=p0.dtype)
    cos_theta[ok] = dot[ok] / dv[ok]
    cos_theta[ok] = np.clip(cos_theta[ok], -1.0, 1.0)
    out[ok] = np.arccos(cos_theta[ok])
    return out

### Helpers

In [10]:
def get_xy_df(df, joint):
    x = pd.to_numeric(df[f"{joint}_x"], errors="coerce").to_numpy(dtype=float)
    y = pd.to_numeric(df[f"{joint}_y"], errors="coerce").to_numpy(dtype=float)
    return np.stack([x, y], axis=1)


def midpoint_strict(a, b):
    ok = np.isfinite(a).all(axis=1) & np.isfinite(b).all(axis=1)
    out = np.full_like(a, np.nan)
    out[ok] = (a[ok] + b[ok]) / 2.0
    return out

### Feature computation

In [11]:
def compute_angle_features_2d_df(df):
    P = {j: get_xy_df(df, j) for j in JOINTS}

    hip_center = midpoint_strict(P["left_hip"], P["right_hip"])
    shoulder_center = midpoint_strict(P["left_shoulder"], P["right_shoulder"])

    y_axis = np.array([0.0, 1.0], dtype=float)
    x_axis = np.array([1.0, 0.0], dtype=float)

    feats = []
    feat_names = []

    # New joints (centers)
    feats += [hip_center[:, 0], hip_center[:, 1]]
    feat_names += ["hip_center_x", "hip_center_y"]

    feats += [shoulder_center[:, 0], shoulder_center[:, 1]]
    feat_names += ["shoulder_center_x", "shoulder_center_y"]

    # 8 joint angles
    feats += [angle_abc(P["left_shoulder"], P["left_elbow"], P["left_wrist"])]
    feat_names += ["ang_left_elbow"]
    feats += [angle_abc(P["right_shoulder"], P["right_elbow"], P["right_wrist"])]
    feat_names += ["ang_right_elbow"]

    feats += [angle_abc(P["left_hip"], P["left_shoulder"], P["left_elbow"])]
    feat_names += ["ang_left_shoulder"]
    feats += [angle_abc(P["right_hip"], P["right_shoulder"], P["right_elbow"])]
    feat_names += ["ang_right_shoulder"]

    feats += [angle_abc(P["left_shoulder"], P["left_hip"], P["left_knee"])]
    feat_names += ["ang_left_hip"]
    feats += [angle_abc(P["right_shoulder"], P["right_hip"], P["right_knee"])]
    feat_names += ["ang_right_hip"]

    feats += [angle_abc(P["left_hip"], P["left_knee"], P["left_ankle"])]
    feat_names += ["ang_left_knee"]
    feats += [angle_abc(P["right_hip"], P["right_knee"], P["right_ankle"])]
    feat_names += ["ang_right_knee"]

    # 12 axis angles
    feats += [angle_segment_to_axis(P["left_hip"], P["left_shoulder"], y_axis)]
    feat_names += ["ang_left_hip_shoulder_y"]
    feats += [angle_segment_to_axis(P["right_hip"], P["right_shoulder"], y_axis)]
    feat_names += ["ang_right_hip_shoulder_y"]

    feats += [angle_segment_to_axis(P["left_shoulder"], P["left_elbow"], y_axis)]
    feat_names += ["ang_left_shoulder_elbow_y"]
    feats += [angle_segment_to_axis(P["right_shoulder"], P["right_elbow"], y_axis)]
    feat_names += ["ang_right_shoulder_elbow_y"]

    feats += [angle_segment_to_axis(P["left_elbow"], P["left_wrist"], y_axis)]
    feat_names += ["ang_left_elbow_wrist_y"]
    feats += [angle_segment_to_axis(P["right_elbow"], P["right_wrist"], y_axis)]
    feat_names += ["ang_right_elbow_wrist_y"]

    feats += [angle_segment_to_axis(P["left_hip"], P["left_knee"], y_axis)]
    feat_names += ["ang_left_hip_knee_y"]
    feats += [angle_segment_to_axis(P["right_hip"], P["right_knee"], y_axis)]
    feat_names += ["ang_right_hip_knee_y"]

    feats += [angle_segment_to_axis(P["left_knee"], P["left_ankle"], y_axis)]
    feat_names += ["ang_left_knee_ankle_y"]
    feats += [angle_segment_to_axis(P["right_knee"], P["right_ankle"], y_axis)]
    feat_names += ["ang_right_knee_ankle_y"]

    feats += [angle_segment_to_axis(P["left_hip"], P["right_hip"], x_axis)]
    feat_names += ["ang_hips_x"]
    feats += [angle_segment_to_axis(P["left_shoulder"], P["right_shoulder"], x_axis)]
    feat_names += ["ang_shoulders_x"]

    # Back y axis angle
    feats += [angle_segment_to_axis(hip_center, shoulder_center, y_axis)]
    feat_names += ["ang_hipcenter_shouldercenter_y"]

    feat_mat = np.stack(feats, axis=1)
    return feat_mat, np.array(feat_names, dtype=object)

### Processing

In [12]:
def process_directory_csv(in_dir, out_dir):
    out_dir.mkdir(parents=True, exist_ok=True)

    files = sorted(in_dir.glob("*.csv"))
    for i, path in enumerate(files, 1):
        df = pd.read_csv(path)

        feat_mat, feat_names = compute_angle_features_2d_df(df)
        for k, name in enumerate(feat_names):
            df[name] = feat_mat[:, k]

        out_path = out_dir / f"{path.stem}.csv"
        df.to_csv(out_path, index=False)

        if i % 500 == 0 or i == len(files):
            print(f"[{in_dir.name}] {i}/{len(files)} done")


process_directory_csv(IN_DIR, OUT_DIR)

[keypoints_combined] 10/10 done
